# ABC MMM — Meridian Scenario Planner (Looker Studio)

End-to-end notebook that:

1. Pulls ABC weekly data from `donut-426.abc.mmm`
2. Re-trains the Meridian MMM with ABC's custom priors + adstock map (matches MMMv11)
3. Writes four scenarios (reallocation, budget flex, response curves, target ROAS) to BigQuery — keeps the existing `donut-426.abc.v_scenario_*` pipeline alive
4. Runs the **Meridian Scenario Planner Beta** to push scenarios into a Google Sheet and emit a pre-built Looker Studio dashboard URL
5. Exports the monthly spend / revenue / ROAS tables MMMv11 produces

**Runtime:** GPU required (T4 free tier is fine). Expect ~25–40 min end-to-end: ~15–25 min for model fit, ~10–15 min for the scenario planner's quarterly inferences.

**Prerequisites:** `scenario_planner_final.py` in `MyDrive/ABC/MMM/` (for the dual-write to BigQuery).

## 0 — Install

In [ ]:
# @markdown Install Meridian with scenario-planner extras. ~5–7 min on first run.
!pip -q install --upgrade google-meridian[scenarioplanner,and-cuda]

## 1 — Imports + auth

In [ ]:
import datetime
import os

import arviz as az
import numpy as np
import pandas as pd
import pandas_gbq
import tensorflow as tf
import tensorflow_probability as tfp

from meridian import constants
from meridian.analysis import analyzer as analyzer_module
from meridian.analysis import optimizer as budget_optimizer_module
from meridian.analysis import summarizer
from meridian.analysis import visualizer
from meridian.data import data_frame_input_data_builder as builder_module
from meridian.model import model as mmm_module
from meridian.model import prior_distribution
from meridian.model import spec

# Scenario planner imports (new — from the scenarioplanner extra)
from meridian.schema.processors import model_fit_processor
from meridian.schema.processors import marketing_processor
from meridian.schema.processors import budget_optimization_processor
from meridian.schema.utils import date_range_bucketing
from scenarioplanner.converters import sheets
from scenarioplanner.converters.dataframe import dataframe_model_converter
from scenarioplanner import mmm_ui_proto_generator as mmm_ui_gen
from scenarioplanner.linkingapi import url_generator

print('GPU available:', len(tf.config.list_physical_devices('GPU')) > 0)
print('TensorFlow:', tf.__version__)

In [ ]:
# @markdown Authenticate with Google — grants this Colab access to BigQuery, Drive, and Sheets.
from google.colab import auth, drive

_SCOPES = [
    'https://www.googleapis.com/auth/bigquery',
    'https://www.googleapis.com/auth/drive',
    'https://www.googleapis.com/auth/spreadsheets',
]
credentials = auth.authenticate_user(_SCOPES)
drive.mount('/content/drive')

## 2 — Pull ABC data from BigQuery

In [ ]:
PROJECT_ID = 'donut-426'
view_sql = "SELECT * FROM abc.mmm WHERE Model_Dates = 'In Model'"

df_bq = pandas_gbq.read_gbq(view_sql, project_id=PROJECT_ID)
df_bq['time'] = pd.to_datetime(df_bq['time'])
print('Loaded ABC weekly data. Shape:', df_bq.shape)

## 3 — Build InputData (9 paid channels, non-media, controls)

In [ ]:
media_channels = [
    'Meta', 'Search', 'PMAX', 'Amex',
    'Video_Epsilon', 'Video_Google', 'Video_Hulu', 'Video_MNTN', 'Video_Paramount',
]
media_impression_cols = [
    'Meta_impression', 'Search_click', 'PMAX_impression', 'Amex_impression',
    'Video_Epsilon_impression', 'Video_Google_impression', 'Video_Hulu_impression',
    'Video_MNTN_impression', 'Video_Paramount_impression',
]
media_spend_cols = [f'{ch}_spend' for ch in media_channels]
non_media_cols = [
    'Vault_Drops', 'Email_sends',
    'Conversions_BasketSize', 'Conversions_Pricing',
    'StoreCount', 'Celebs',
]
control_cols = ['Category_Interest', 'Hurricanes', 'LightningSales']
population_col = 'population' if 'population' in df_bq.columns else None

required = ['time', 'geo', 'Conversions_Revenue'] + media_impression_cols + media_spend_cols + non_media_cols + control_cols
missing = [c for c in required if c not in df_bq.columns]
if missing:
    raise ValueError(f'Missing columns: {missing}')

builder = builder_module.DataFrameInputDataBuilder(
    kpi_type='revenue',
    default_kpi_column='Conversions_Revenue',
)
builder = builder.with_kpi(df_bq)
builder = builder.with_media(
    df_bq,
    media_cols=media_impression_cols,
    media_spend_cols=media_spend_cols,
    media_channels=media_channels,
)
if population_col:
    builder = builder.with_population(df_bq, population_col=population_col)
builder = builder.with_non_media_treatments(df_bq, non_media_treatment_cols=non_media_cols)
builder = builder.with_controls(df_bq, control_cols=control_cols)
input_data = builder.build()
print('InputData built.')

## 4 — Priors, adstock map, model spec (matches MMMv11)

In [ ]:
# ROI priors — Search tightest/most centered, Video group centered ~5.2×
roi_mu = tf.constant(
    [1.40, 1.35, 1.15, 1.15, 1.65, 1.65, 1.65, 1.65, 1.65], dtype=tf.float32
)
roi_sigma = tf.constant(
    [0.40, 0.28, 0.45, 0.45, 0.25, 0.25, 0.25, 0.25, 0.25], dtype=tf.float32
)
roi_prior = tfp.distributions.LogNormal(loc=roi_mu, scale=roi_sigma)

# Adstock alpha — Beta
alpha_conc1 = tf.constant([2.0] * 4 + [3.0] * 5, dtype=tf.float32)
alpha_conc0 = tf.constant([2.0] * 9, dtype=tf.float32)
alpha_prior = tfp.distributions.Beta(concentration1=alpha_conc1, concentration0=alpha_conc0)

# Hill ec — TruncatedNormal
ec_mu = tf.constant([1.0] * 9, dtype=tf.float32)
ec_sigma = tf.constant([0.5] * 4 + [0.30] * 5, dtype=tf.float32)
ec_prior = tfp.distributions.TruncatedNormal(loc=ec_mu, scale=ec_sigma, low=0.1, high=10.0)

prior = prior_distribution.PriorDistribution(
    roi_m=roi_prior,
    alpha_m=alpha_prior,
    ec_m=ec_prior,
)

adstock_map = {
    'Search': 'geometric',
    'Meta': 'geometric',
    'PMAX': 'geometric',
    'Amex': 'geometric',
    'Video_Epsilon': 'binomial',
    'Video_Google': 'binomial',
    'Video_Hulu': 'binomial',
    'Video_MNTN': 'binomial',
    'Video_Paramount': 'binomial',
}

model_spec = spec.ModelSpec(
    prior=prior,
    media_effects_dist='log_normal',
    adstock_decay_spec=adstock_map,
    max_lag=12,
    hill_before_adstock=True,
)
print('Model spec ready.')

## 5 — Fit model (~15–25 min on T4)

In [ ]:
mmm = mmm_module.Meridian(input_data=input_data, model_spec=model_spec)
mmm.sample_prior(500)
mmm.sample_posterior(
    n_chains=20,
    n_adapt=3000,
    n_burnin=1000,
    n_keep=3000,
    seed=0,
    dual_averaging_kwargs={'target_accept_prob': 0.85},
)
print('Posterior sampled.')

# Quick fit check — full diagnostics live in MMMv11; here we just confirm R-hat
print(az.summary(mmm.inference_data, var_names=['roi_m'])[['mean', 'r_hat', 'ess_bulk']])

## 6 — Dual-write: four scenarios to BigQuery (`donut-426.abc.v_scenario_*`)

In [ ]:
# @markdown Runs `scenario_planner_final.py` from Drive — writes the 4 scenarios
# @markdown (reallocation, budget flex, response curves, target ROAS) to BigQuery.
# @markdown The Looker Studio views under `donut-426.abc.v_scenario_*` always point at the latest run_id.
SCENARIO_SCRIPT = '/content/drive/MyDrive/ABC/MMM/scenario_planner_final.py'

if not os.path.exists(SCENARIO_SCRIPT):
    raise FileNotFoundError(
        f'{SCENARIO_SCRIPT} not found. Upload scenario_planner_final.py to MyDrive/ABC/MMM/ '
        'or change SCENARIO_SCRIPT above.'
    )

exec(open(SCENARIO_SCRIPT).read(), globals())
print('Scenarios written to BigQuery.')

## 7 — Meridian Scenario Planner: push scenarios to Google Sheets + Looker Studio URL

This is the new path. One cell runs the full scenario inference (quarterly breakdowns), uploads everything to a Google Sheet, and emits a URL that opens a pre-built Looker Studio dashboard wired to that sheet.

Tweak `min_spend_shift_ratio` / `max_spend_shift_ratio` to widen the interactive optimizer's range on the dashboard. Default 0.3/0.3 = ±30% per channel (matches the BQ budget-flex sweep).

In [ ]:
# @title Scenario planner config

spreadsheet_name = 'ABC MMM Scenario Planner — FY26'  # @param {"type":"string"}
optimization_name = 'ABC FY26'  # @param {"type":"string"}
include_non_paid_channels = True  # @param {"type":"boolean"}

# Time breakdowns — generates separate optimizations per bucket. Quarterly matches ABC cadence.
yearly = False  # @param {"type":"boolean"}
quarterly = True  # @param {"type":"boolean"}
monthly = False  # @param {"type":"boolean"}

# Per-channel spend bounds (as proportions). 0.3 = ±30%, matches the BQ budget-flex sweep.
min_spend_shift_ratio = 0.3  # @param {"type":"raw"}
max_spend_shift_ratio = 0.3  # @param {"type":"raw"}

# Reach & frequency — ABC has no R&F channels today, but leave the defaults in case we add them.
use_optimal_frequency = True  # @param {"type":"boolean"}
max_frequency = 10.0  # @param {"type":"raw"}

time_breakdown_generators = []
if yearly:
    time_breakdown_generators.append(date_range_bucketing.YearlyDateRangeGenerator)
if quarterly:
    time_breakdown_generators.append(date_range_bucketing.QuarterlyDateRangeGenerator)
if monthly:
    time_breakdown_generators.append(date_range_bucketing.MonthlyDateRangeGenerator)

channel_constraints = [
    budget_optimization_processor.ChannelConstraintRel(
        channel_name=ch,
        spend_constraint_lower=min_spend_shift_ratio,
        spend_constraint_upper=max_spend_shift_ratio,
    )
    for ch in mmm.input_data.get_all_paid_channels()
]

budget_opt_spec = budget_optimization_processor.BudgetOptimizationSpec(
    start_date=None,
    end_date=None,
    optimization_name=optimization_name,
    grid_name='-'.join(optimization_name.lower().split(' ')),
    constraints=channel_constraints,
    use_optimal_frequency=use_optimal_frequency,
    max_frequency=max_frequency,
)

print('Running scenario inference — this can take 10–15 min.')
mmm_proto = mmm_ui_gen.create_mmm_ui_data_proto(
    mmm=mmm,
    specs=[
        model_fit_processor.ModelFitSpec(),
        marketing_processor.MarketingAnalysisSpec(
            media_summary_spec=marketing_processor.MediaSummarySpec(
                include_non_paid_channels=include_non_paid_channels,
            ),
        ),
        budget_opt_spec,
    ],
    time_breakdown_generators=time_breakdown_generators,
)

print('Converting to dataframes.')
dataframes = dataframe_model_converter.DataFrameModelConverter(mmm_proto)()

print('Uploading to Google Sheets.')
spreadsheet = sheets.upload_to_gsheet(
    dataframes, credentials, spreadsheet_name=spreadsheet_name
)
print(f'Spreadsheet URL: {spreadsheet.url}')

In [ ]:
# @markdown Generate the Looker Studio dashboard URL
from IPython.display import HTML

report_url = url_generator.create_report_url(spreadsheet)
HTML(f'<a href="{report_url}" target="_blank">Open ABC MMM Scenario Planner in Looker Studio</a>')

## 8 — Monthly spend / revenue / ROAS extract (matches MMMv11)

Writes `monthly_spend_roas_by_channel.csv` + `monthly_spend_roas_table.html` to `MyDrive/ABC/MMM/` — same output as the tail of MMMv11.

In [ ]:
OUT_DIR = '/content/drive/MyDrive/ABC/MMM'
os.makedirs(OUT_DIR, exist_ok=True)

analyzer = analyzer_module.Analyzer(mmm)

# Monthly spend (actuals)
df_monthly = df_bq.copy()
df_monthly['month'] = df_monthly['time'].dt.to_period('M').dt.to_timestamp()
monthly_spend = df_monthly.groupby('month')[media_spend_cols].sum().reset_index()
monthly_spend = monthly_spend.melt(
    id_vars='month', value_vars=media_spend_cols,
    var_name='channel_spend', value_name='spend',
)
monthly_spend['channel'] = monthly_spend['channel_spend'].str.replace('_spend', '')

# Modeled incremental revenue by channel × time
print('Calculating incremental outcomes …')
inc_tensor = analyzer.incremental_outcome(
    aggregate_geos=True,
    aggregate_times=False,
    include_non_paid_channels=False,
)
inc_mean = np.mean(inc_tensor.numpy(), axis=(0, 1))

inc_by_time = pd.DataFrame(inc_mean, columns=media_channels)
inc_by_time['time'] = pd.to_datetime(df_bq['time'].values)
inc_by_time['month'] = inc_by_time['time'].dt.to_period('M').dt.to_timestamp()
inc_by_time = inc_by_time.melt(id_vars=['time', 'month'], var_name='channel', value_name='mean')

monthly_roas = inc_by_time.groupby(['month', 'channel'])['mean'].sum().reset_index()
monthly_roas = monthly_roas.merge(monthly_spend[['month', 'channel', 'spend']], on=['month', 'channel'])
monthly_roas['roas'] = monthly_roas['mean'] / monthly_roas['spend']

monthly_table = monthly_roas.pivot(index='month', columns='channel', values=['spend', 'roas', 'mean'])
monthly_table.columns = ['_'.join(col).strip() for col in monthly_table.columns.values]
monthly_table = monthly_table.reset_index()

monthly_table.to_csv(f'{OUT_DIR}/monthly_spend_roas_by_channel.csv', index=False)
with open(f'{OUT_DIR}/monthly_spend_roas_table.html', 'w') as fh:
    fh.write(monthly_table.to_html(index=False, float_format='%.2f'))
print(f'Monthly tables saved to {OUT_DIR}')
monthly_table.head()

## 9 — Save & share the Looker Studio dashboard

When you click the Looker Studio link from Section 7 the first time:

1. Hit **Edit and save** at the top, then **Acknowledge and save**.
2. Click **Allow** to enable community visualizations (one-time per Google account).
3. Refresh the page if charts don't render immediately.
4. Share the saved report view-only with the ABC marketing team. They can tweak per-channel spend and see optimized ROAS + revenue live.

**To refresh** after weekly data lands: re-run the notebook end-to-end. The BQ views under `donut-426.abc.v_scenario_*` auto-point at the latest `run_id`, and a new Sheet (with new timestamp suffix) gets created for the Meridian dashboard — overwrite the existing sheet or create a new one depending on whether you want history.